# 实验二 · 向量加法 Add —— 三段流水编程范式与多核切分

**所属**：《并行计算》第六章 · 昇腾 Ascend C 算子开发　|　**难度**：⭐⭐⭐ 进阶　|　**预计时长**：50–60 分钟

实验一只启动了核函数而未搬运任何数据。本实验实现第一个真正的算子：逐元素向量加法。其数学定义极其简单，但要在昇腾 AI Core 上正确且高效地完成它，必须回答三个问题：数据如何在 Global Memory 与片上缓冲之间流动、多个核如何分工、以及搬运与计算如何重叠。四个版本依次回答这三个问题，并处理最后一个工程性问题——任意长度输入的正确性。

> **实验说明**
> 1. 本实验的核心内容有四点：三段流水编程范式、SPMD 多核数据切分、双缓冲、以及非整除长度下的尾核与尾块处理。这四点构成了后续全部矢量算子的基本框架。
> 2. 本实验采用**递进式的版本组织**：以单核实现为基准，每个版本仅引入一个新概念，便于对照阅读。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 5. 四个版本的核函数、CPU 基准、数据生成、结果校验与 `main` 都写在同一个 `.asc` 文件中，由一条 `bisheng` 命令编译为单个可执行程序。
> 6. 输入数据在程序内部生成（固定随机种子），不读写任何二进制文件；参考值由 CPU 基准实现顺带算出，既作为校验依据，也作为性能基准。
> 7. 本实验建立在**实验一 HelloWorld** 的基础之上，建议先完成实验一。

## 🎯 学习目标

完成本实验后，开发者应能够：

- 说明矢量计算单元为何不能直接访问 Global Memory，以及这与 CPU 缓存的本质区别
- 掌握 `GlobalTensor`、`LocalTensor`、`TPipe`、`TQue` 四类对象各自的职责
- 掌握 **CopyIn / Compute / CopyOut** 三段流水编程范式，能独立搭建算子框架
- 说明 `AllocTensor`、`EnQue`、`DeQue`、`FreeTensor` 四个接口的同步语义
- 掌握 SPMD 多核数据切分方法，能依据 `block_idx` 计算数据分片的偏移与长度
- 说明 `KERNEL_TASK_TYPE_DEFAULT` 的作用，以及缺少该声明会对核数扫描实验产生什么影响
- 理解**双缓冲**掩盖搬运延迟的原理，能核算其片上缓冲代价并对照 UB 容量判断可行性
- 掌握**两级切分**：核间切分决定并行度，核内分块受片上缓冲容量约束
- 掌握非整除长度下的尾核与尾块处理方法，能说明 32 字节对齐约束的来源
- 正确测量核函数耗时：预热、多次重复取平均、先同步再停止计时，并说明基准实现须满足的条件
- 说明为何不以「单个算子的主机—设备往返耗时」评价卸载收益，该判断的单位为何是整段计算图
- 基于实测数据构建版本对照表，用 Roofline 模型判断本算子的瓶颈所在

## 🗺️ 学习路径

1. **准备阶段**：理解片上缓冲的显式管理，以及由此产生的分块需求
2. **Device 侧实现**：三个算子类依次实现单核基线、多核与双缓冲、变长与尾块
3. **Host 侧实现**：数据生成、CPU 基准、结果校验与统一计时
4. **编译运行**：一条 `bisheng` 命令，一次运行产出全部对照数据
5. **结果可视化**：以 CPU 与 v1 两套基线分别计算加速比
6. **参数扫描**：分别扫描核数、分块长度与数据规模，定位各自的作用区间
7. **结果分析**：用 Roofline 模型判断本算子的瓶颈所在

## 1. 背景与动机：为什么必须分块

第二章讨论过存储层次结构。在 CPU 上，数据从主存到寄存器的搬运由硬件缓存自动完成。缓存未命中只会降低性能，不会影响程序的正确性——开发者通过调整访问顺序改善局部性，但并不直接控制数据所在的位置。

昇腾 AI Core 的存储组织方式不同。**矢量计算单元只能访问片上的统一缓冲区（Unified Buffer, UB），无法直接读写 Global Memory。** 数据必须由搬运单元显式地搬入片上缓冲，计算完成后再显式搬回。

<img src="images/06.02_memory_compare.png" alt="CPU 缓存与 AI Core 片上缓冲的对比" width="800">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 对比项 | CPU 缓存 | AI Core 片上缓冲（UB） |
| --- | --- | --- |
| 管理方式 | 硬件自动 | **软件显式** |
| 数据不在其中的后果 | 性能下降 | **无法计算** |
| 开发者的控制手段 | 调整访问顺序 | 直接编写搬运指令 |
| 容量 | MB 级（L2/L3） | **192 KB / 核** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对比项</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">CPU 缓存</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">AI Core 片上缓冲（UB）</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">管理方式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">硬件自动</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>软件显式</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数据不在其中的后果</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">性能下降</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>无法计算</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">开发者的控制手段</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">调整访问顺序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">直接编写搬运指令</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">容量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">MB 级（L2/L3）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>192 KB / 核</strong></td>
</tr>
</tbody>
</table>

本实验的单个张量为 8 MB，而 UB 只有 192 KB，相差约 42 倍。因此数据**必须**切分为若干分块，逐块完成「搬入 — 计算 — 搬出」。这不是一项优化，而是能否运行的前提。

### 算法与数据规格

$$ z_i = x_i + y_i, \quad i = 0, 1, \dots, N-1 $$

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 张量 | 形状 | 类型 | 格式 |
| --- | --- | --- | --- |
| 输入 `x` | [2^21] | float | ND |
| 输入 `y` | [2^21] | float | ND |
| 输出 `z` | [2^21] | float | ND |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">张量</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">形状</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">类型</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">格式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入 <code>x</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">[2^21]</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">float</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">ND</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入 <code>y</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">[2^21]</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">float</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">ND</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出 <code>z</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">[2^21]</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">float</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">ND</td>
</tr>
</tbody>
</table>

取 N = 2^21 = 2 097 152，单个张量占用 8 MB。选择 `float` 而非 `half`，是为了沿用前五章的精度校验口径。

> **与第三章 NEON 的对照**：第三章使用 `vaddq_f32(a, b)`，一条指令处理 4 个 float，操作数位于**寄存器**中。本实验的 `AscendC::Add(z, x, y, len)` 是同一思想在更大粒度上的体现——矢量计算单元一次迭代处理 256 字节，即 64 个 float32——但操作数位于 **UB** 中，需要开发者自行安排搬运。第三章处理的向量宽度为 16 字节，本实验的搬运与计算粒度为 **32 字节**；第三章的尾元素处理，在这里变为**尾块处理**（v4）。同一类问题，出现在不同的层级上。

## 2. 三段流水编程范式

### 2.1 四类数据对象

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 对象 | 职责 |
| --- | --- |
| `GlobalTensor<T>` | 描述位于 Global Memory 上的数据，经 `SetGlobalBuffer` 绑定起始地址与长度 |
| `LocalTensor<T>` | 描述位于片上缓冲（UB）的数据，是矢量计算接口直接操作的对象 |
| `TPipe` | 片上内存的统一管理者，经 `InitBuffer` 为队列分配缓冲。一个核函数只能有一个 |
| `TQue` | 流水任务之间的队列，**同时**承担数据传递与同步两项职责 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对象</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">职责</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GlobalTensor&lt;T&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">描述位于 Global Memory 上的数据，经 <code>SetGlobalBuffer</code> 绑定起始地址与长度</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>LocalTensor&lt;T&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">描述位于片上缓冲（UB）的数据，是矢量计算接口直接操作的对象</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TPipe</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">片上内存的统一管理者，经 <code>InitBuffer</code> 为队列分配缓冲。一个核函数只能有一个</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TQue</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">流水任务之间的队列，<strong>同时</strong>承担数据传递与同步两项职责</td>
</tr>
</tbody>
</table>

<img src="images/06.02_memory_management.png" alt="06.02_memory_management" width="700px">

`TPipe` 管理 UB 这一整块空间，`InitBuffer` 从中划分出若干队列，每个队列再划分为若干块缓冲。`AllocTensor` 与 `FreeTensor` 分别对应从队列中取用与归还。**二者必须严格成对**：遗漏 `FreeTensor` 的后果不仅是内存泄漏导致的性能下降，而且后续 `AllocTensor` 永远等不到可用缓冲，核函数无法正常返回。

### 2.2 三个流水任务

核内处理逻辑划分为三段：

<img src="images/06.02_pipeline_tasks.png" alt="06.02_pipeline_tasks" width="800px">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 流水任务 | 职责 | 使用的硬件部件 |
| --- | --- | --- |
| **CopyIn** | 将输入数据由 Global Memory 搬入 UB，完成后入队 | 搬运单元 MTE2 |
| **Compute** | 出队取得输入，调用矢量计算接口，结果入队 | 矢量计算单元 VEC |
| **CopyOut** | 出队取得结果，搬回 Global Memory，随后释放缓冲 | 搬运单元 MTE3 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">流水任务</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">职责</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">使用的硬件部件</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>CopyIn</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">将输入数据由 Global Memory 搬入 UB，完成后入队</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">搬运单元 MTE2</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>Compute</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">出队取得输入，调用矢量计算接口，结果入队</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矢量计算单元 VEC</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>CopyOut</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">出队取得结果，搬回 Global Memory，随后释放缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">搬运单元 MTE3</td>
</tr>
</tbody>
</table>

三者分别在**三条相互独立的指令队列**上排队。正因为相互独立，它们才可能并行执行——这是 2.4 节双缓冲的物理基础。

把三个流水任务放到时间轴上，可以看到它们处理不同数据切片的过程是交错推进的：

<img src="images/06.02_pipeline_running.png" alt="06.02_pipeline_running" width="720px">

图中同一列上下相邻的方框属于不同的数据切片。任务之间的箭头表达数据依赖：`CopyIn` 处理完第一个切片之后，`Compute` 才能对该切片进行处理。要让这种交错真正发生，队列中必须有不止一块缓冲，这正是 §2.4 的内容。

内部的调用关系是固定的：核函数实例化算子类，`Init` 完成初始化，`Process` 循环调用三个流水任务。

<img src="images/06.02_kernel_call_relation.png" alt="06.02_kernel_call_relation" width="620px">

*来源：《Ascend C 算子开发指南 01 入门教程》 图 1-2 核函数调用关系图*

### 2.3 队列的四个基本操作

<img src="images/06.02_queue.png" alt="queue" width="800px">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 接口 | 数据语义 | 同步语义 |
| --- | --- | --- |
| `AllocTensor<T>()` | 从队列申请一块片上缓冲 | 等待该缓冲被下游释放 |
| `EnQue(tensor)` | 将缓冲入队 | **通知下游**：数据已就绪 |
| `DeQue<T>()` | 从队列取出缓冲 | **等待上游**：数据是否就绪 |
| `FreeTensor(tensor)` | 归还缓冲 | **通知上游**：该缓冲可被复用 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">接口</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">数据语义</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">同步语义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>AllocTensor&lt;T&gt;()</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">从队列申请一块片上缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">等待该缓冲被下游释放</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>EnQue(tensor)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">将缓冲入队</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>通知下游</strong>：数据已就绪</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>DeQue&lt;T&gt;()</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">从队列取出缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>等待上游</strong>：数据是否就绪</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>FreeTensor(tensor)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">归还缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>通知上游</strong>：该缓冲可被复用</td>
</tr>
</tbody>
</table>

**与第四章的对照**：这一机制与生产者-消费者模型完全同构。队列在其中同时充当数据通道与同步原语，`EnQue` 与 `DeQue` 的作用相当于条件变量上的通知与等待，`AllocTensor` 与 `FreeTensor` 则对应有界缓冲区的「取空位」与「还空位」。

**关键差异**：第四章中通信双方是两个线程，同步由操作系统完成；此处通信双方是同一个核内的**两类硬件部件**（搬运单元与矢量计算单元），同步由硬件事件完成，开发者不必手工插入同步指令。这正是 TPipe/TQue 框架相对于底层接口的价值所在。

### 2.4 双缓冲：搬运与计算的重叠

若每个队列只有一块缓冲，`CopyIn` 必须等待上一轮的 `Compute` 归还缓冲才能继续，三类部件中始终有两类处于空闲状态。

<img src="images/06.02_double_buffer.png" alt="单缓冲与双缓冲的时间线对照" width="1000">

将缓冲块数改为 2 之后，矢量单元处理其中一块的同时，搬运单元可以向另一块写入下一分块的数据。实现方式很简单：把 `InitBuffer` 的第二个参数由 1 改为 2。

从数据切分的角度看，双缓冲相当于在核内分块之下再切一层：

<img src="images/06.02_double_buffer_split.png" alt="06.02_double_buffer_split" width="900px">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 章节 | 技术 | 重叠的对象 |
| --- | --- | --- |
| 第一章 | 指令流水线 | 一条指令的取指、译码、执行等阶段 |
| 第三章 | 多累加变量 | 多条相互独立的 FMA 指令 |
| **本章** | **双缓冲** | **搬运单元与矢量计算单元两类不同的硬件部件** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">章节</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">技术</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">重叠的对象</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第一章</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">指令流水线</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一条指令的取指、译码、执行等阶段</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第三章</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多累加变量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多条相互独立的 FMA 指令</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>本章</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>双缓冲</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>搬运单元与矢量计算单元两类不同的硬件部件</strong></td>
</tr>
</tbody>
</table>

三者原理相同——通过增加在途工作项的数量掩盖延迟——但重叠的层次逐级上升。

> **两个易混淆的概念**：`InitBuffer(que, n, bytes)` 的第二个参数 `n` 是**缓冲块数**，`TQue<pos, D>` 模板的第二个参数 `D` 是**队列深度**。前者是运行时值，后者是编译期常量。本实验的 v2 与 v3 正是依靠这一差别，用同一份算子类代码实现单缓冲与双缓冲两种行为。
>
> 另外，官方 `add_custom.asc` 采用**固定 UB 预算**的写法：`tileLength = blockLength / tileNum / BUFFER_NUM`，`loopCount = tileNum * BUFFER_NUM`，开启双缓冲时把每块切一半、UB 占用不变。本实验采用**固定分块长度、UB 占用翻倍**的写法，使 v2 与 v3 的差异收敛到一个参数。两种写法等价，阅读官方代码时请注意区分。由此带来的影响见 §13 ②。

## 3. 两级切分与分块参数

数据切分分为两级，二者的约束来源完全不同：

<img src="images/06.02_multicore_and_tiling.png" alt="06.02_multicore_and_tiling" width="900px">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 级别 | 切分依据 | 约束来源 | 决定什么 |
| --- | --- | --- | --- |
| 核间切分 | `block_idx` | 可用核数 | 并行度 |
| 核内分块 | 循环下标 | **片上缓冲容量** | 单次搬运的数据量 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">级别</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">切分依据</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">约束来源</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">决定什么</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核间切分</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>block_idx</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可用核数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">并行度</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核内分块</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">循环下标</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>片上缓冲容量</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单次搬运的数据量</td>
</tr>
</tbody>
</table>

**第一级：多核并行。** 用 `AscendC::GetBlockIdx()` 取得当前核的编号，通过 `blockLength * GetBlockIdx()` 计算 GM 上的地址偏移，确保各核处理互不重叠的数据段。

<img src="images/06.02_multicore_split.png" alt="06.02_multicore_split" width="820px">

**第二级：核内分块。** 单核上的数据继续切分为若干个 `TILE_LENGTH` 大小的块，循环处理。

<img src="images/06.02_single_core_tiling.png" alt="06.02_single_core_tiling" width="820px">

### 3.1 片上缓冲占用的核算

核内同时存在三个缓冲区，分别对应 x、y、z。总占用为：

> 占用 = 3 × TILE_LENGTH × sizeof(float) × BUFFER_NUM

本实验取 `TILE_LENGTH = 4096`（16 KB），则：（下表中的 v1 至 v4 是本实验的四个递进版本，其划分见 §6）

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| BUFFER_NUM | 占用 | 占 UB（192 KB）的比例 |
| --- | --- | --- |
| 1（v1、v2） | 3 × 16 KB × 1 = 48 KB | 25 % |
| 2（v3、v4） | 3 × 16 KB × 2 = 96 KB | **50 %** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">BUFFER_NUM</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">占用</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">占 UB（192 KB）的比例</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1（v1、v2）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3 × 16 KB × 1 = 48 KB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">25 %</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2（v3、v4）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3 × 16 KB × 2 = 96 KB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>50 %</strong></td>
</tr>
</tbody>
</table>

> 这项核算应当在编写代码之前完成。修改分块长度或缓冲块数时都要重新计算。占用超出 UB 容量将导致编译失败或运行异常；即使核算结果恰好不超出，占用逼近容量上限时也会因为编译器缺少临时空间而影响可编译性与性能。§12.2 的分块长度扫描会给出逼近这条边界时的实测表现。

### 3.2 对齐约束

Global Memory 与片上缓冲之间的搬运以 **32 字节**为基本单位。对 `float` 而言，即每次搬运的起始地址与长度均须为 **8 个元素**的整数倍。

<img src="images/06.02_tail_block.png" alt="06.02_tail_block" width="900px">

*来源：《Ascend C 算子开发指南 03 算子实践参考》 图 3-14 多核 Tiling 尾块示意图。图中 `lastTileLength` 即核内的尾块长度*

本实验约定元素总数为 8 的整数倍，并在 v4 中通过每核基础长度向下对齐、由末核承担余量的方式，保证各核的起始地址同样满足对齐要求。

元素总数本身不是 8 的整数倍时，官方的做法是先把总长度**向上**对齐到 32 字节的整数倍，再按对齐后的长度做多核切分，多出来的部分在搬运与计算时另行处理。完整方案见官方的非对齐场景章节，本实验不展开，相关线索留在思考题第 5 题。

> **术语说明**：官方把核间与核内的两类余量分别称为**尾核**与**尾块**。
>
> - **尾块**指核内最后不足一个 `TILE_LENGTH` 的部分，官方在 Tiling 结构体中用 `lastTileLength` 描述。
> - **尾核**的官方定义需要特别注意：当数据无法均分到各核时，官方要求**尽可能均匀地分配**，把计算量较多的核称为**整核**（`formerNum` / `formerLength`），计算量较少的核称为**尾核**（`tailNum` / `tailLength`），二者满足 `formerNum × formerLength + tailNum × tailLength = totalLengthAligned`。也就是说，**官方的尾核是算得少的那一组核，数量可以多于一个**。
>
> 下图是官方给出的例子：待处理数据无法均分到 8 个核上，前 5 个核各算 16 个 datablock（整核），后 3 个核各算 15 个（尾核）。
>
> <img src="images/06.02_tail_core.png" alt="06.02_tail_core" width="900px">
>
> 本实验 v4 采用的是一种**简化方案**：每核基础长度向下对齐，把全部余量交给最后一个核。它同样不遗漏元素、同样保持对齐，代码也短得多，但与官方方案有两点不同：末核是算得**最多**的核，且余量集中在一个核上而非均摊。在本实验的规模下余量不超过数十个元素，负载差异可以忽略；当余量较大时应改用官方的整核／尾核方案。

## 4. 环境准备与检查

In [ ]:
!mkdir -p src_add

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")

In [ ]:
import shutil, subprocess

print("bisheng  :", shutil.which("bisheng") or "⚠️  未找到，请重新执行上一个单元格")
print("npu-smi  :", shutil.which("npu-smi") or "⚠️  未找到")
if shutil.which("npu-smi"):
    print()
    print(
        subprocess.run(["npu-smi", "info"], capture_output=True, text=True).stdout[:1800]
    )

## 5. 本实验的性能测量方法

### 5.1 两个性能指标

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 指标 | 计时范围 | 反映的问题 |
| --- | --- | --- |
| `cpu_ms` | CPU 单线程纯计算循环 | 基准耗时 |
| `npu_kernel_ms` | `<<<>>>` 下发至 `aclrtSynchronizeStream` 返回 | NPU 侧执行该算子的耗时 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">指标</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">计时范围</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">反映的问题</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>cpu_ms</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CPU 单线程纯计算循环</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基准耗时</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>npu_kernel_ms</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>&lt;&lt;&lt;&gt;&gt;&gt;</code> 下发至 <code>aclrtSynchronizeStream</code> 返回</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">NPU 侧执行该算子的耗时</td>
</tr>
</tbody>
</table>

本实验只测量上述两个指标，不单独测量「主机拷贝 + 核函数 + 设备回传」这一端到端口径。原因在于该口径所描述的调用方式并不成立：真实应用中输入数据一次搬入设备内存后会常驻其上，接连由多个算子处理，直至整段计算结束才回传一次结果，主机与设备之间的搬运开销由整段计算图共同摊薄。若为每个算子各安排一次完整往返再计时，测得的是一种实际不会采用的用法；由此得到的加速比既随算子的计算访存比而变，也无法回答「该算子是否值得放到 NPU 上执行」——这一判断的单位是整段计算图，而不是单个算子。相关分析见 §13 ④。

### 5.2 测量要点

1. **预热**。第一次调用核函数包含二进制加载等一次性开销；CPU 侧第一遍也会因缺页而变慢。
2. **多次重复取平均**。单次测量的抖动可达数十个百分点。
3. **先同步、再停止计时**。`<<<>>>` 是异步接口，若不调用 `aclrtSynchronizeStream` 就停止计时，测得的是任务下发耗时（几微秒），而非核函数的执行耗时。
4. **基准实现与被测实现使用相同的优化级别**。本实验的 CPU 基准与 NPU 代码在同一次 `bisheng -O2` 中编译。若基准使用 `-O0`，其耗时会被人为放大，加速比随之失真。
5. **基准实现的结果必须被使用**。若 CPU 侧算出的结果没有被后续代码引用，`-O2` 下整段循环可能被编译器消除。本实验的做法是让 CPU 基准的输出直接作为校验用的参考值。

此外，性能测量的代码路径上不应出现 `AscendC::printf`，设备侧打印的开销可达核函数本身的数十倍。

## 6. 版本设计总览

四个版本的核函数全部写在同一个 `.asc` 文件中，共用同一份 Host 侧代码、同一组数据与同一套计时逻辑，因此版本之间的差异只体现在核函数本身，性能数据具有可比性。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 新增的唯一概念 | 算子类 | 核数 | 缓冲块数 | UB 占用 | 长度来源 |
| --- | --- | --- | --- | --- | --- | --- |
| **v1** | 三段流水框架与核内分块循环 | `KernelAddSingleCore` | 1 | 1 | 48 KB (25 %) | 编译期常量 |
| **v2** | SPMD 多核数据切分 | `KernelAddMultiCore` | 8 | 1 | 48 KB (25 %) | 编译期常量 |
| **v3** | 双缓冲 | `KernelAddMultiCore` | 8 | **2** | **96 KB (50 %)** | 编译期常量 |
| **v4** | 变长输入与尾核尾块处理 | `KernelAddVarLength` | 8 | 2 | 96 KB (50 %) | **运行时参数** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">新增的唯一概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">算子类</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">缓冲块数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">UB 占用</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">长度来源</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三段流水框架与核内分块循环</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelAddSingleCore</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">48 KB (25 %)</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译期常量</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v2</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">SPMD 多核数据切分</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelAddMultiCore</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">48 KB (25 %)</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译期常量</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v3</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">双缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelAddMultiCore</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>2</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>96 KB (50 %)</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译期常量</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v4</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">变长输入与尾核尾块处理</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelAddVarLength</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">96 KB (50 %)</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>运行时参数</strong></td>
</tr>
</tbody>
</table>

两条线索：

- **v1 → v2**：增加并行度。数据被切分到多个核上，属于**核间**层次的改进。
- **v2 → v3**：提高单核内的部件利用率。数据划分不变，改变的是搬运与计算的重叠方式，属于**核内**层次的改进。

v4 不以性能为目标，而是解决工程适用性问题。它把元素总数由编译期常量改为运行时参数。

**三个算子类的对应关系**：v2 与 v3 的算子类逐字相同，唯一差别是缓冲块数，因此共用 `KernelAddMultiCore` 一个类，由 `Init` 的参数区分；v1 与 v4 各有一个类。下面的代码按 **Device 侧三个算子类 + 核函数入口** 与 **Host 侧基础设施 + 主程序** 的顺序写入同一个文件。

## 7. Device 侧实现

官方把矢量算子核函数的实现归纳为三步：算子分析、核函数定义、按矢量编程范式实现算子类。第三步展开即为 CopyIn、Compute、CopyOut 三个流水任务。

<img src="images/06.02_vector_kernel_flow.png" alt="06.02_vector_kernel_flow" width="640px">

本节的四个版本共用这一框架，差异只在算子类内部。

### 7.1 头文件与参数常量

所有参数都写成可被命令行覆盖的宏：

```cpp
#ifndef LAB_TILE_LENGTH
#define LAB_TILE_LENGTH (4 * 1024)
#endif
```

这样 §12 做参数扫描时，只要在编译命令后追加 `-DLAB_TILE_LENGTH=1024` 即可切换，无需修改源码，也无需重建工程。

`ALIGN_ELEM = 32 / sizeof(float) = 8` 是 §3.2 对齐约束的代码表达。`QUEUE_DEPTH` 与 `BUFFER_NUM` 的区别见 §2.4。

> 第一个代码单元格使用 `%%writefile`（**覆盖创建**），其余均为 `%%writefile -a`（**追加**）。修改代码后需要从本单元格开始按顺序重新执行。

In [ ]:
%%writefile src_add/ascendc_vector_add.asc
/**
 * 并行计算 第六章 实验二：向量加法 Add
 *
 * 本文件包含四个版本的核函数、CPU 基准、数据生成、精度校验与 main，
 * 由一条 bisheng 命令编译为单个可执行程序：
 *
 *   bisheng src_add/ascendc_vector_add.asc --npu-arch=dav-2201 -O2 -o src_add/ascendc_vector_add
 *
 * 用法：
 *   ./ascendc_vector_add            四个版本对照，N 取编译期常量 TOTAL_LENGTH
 *   ./ascendc_vector_add <N>        仅运行 v4，N 为任意 8 的整数倍（用于变长与规模扫描）
 */
#include <cstdio>
#include <cstdint>
#include <cstdlib>
#include <cmath>
#include <ctime>  // 计时：clock_gettime
#include <vector>
#include <algorithm>

#include "acl/acl.h"          // Host 侧
#include "kernel_operator.h"  // Device 侧

/* ===================== 可由命令行 -D 覆盖的参数 ===================== */

using LabDType = float; /* 采用 float，与前五章的精度校验口径一致 */

/* 元素总数：2^21 = 2 097 152，单个张量 8 MB，远大于 UB(192KB)，必须分块 */
#ifndef LAB_TOTAL_LENGTH
#define LAB_TOTAL_LENGTH (2 * 1024 * 1024)
#endif
constexpr uint32_t TOTAL_LENGTH = static_cast<uint32_t>(LAB_TOTAL_LENGTH);

/* 分块长度：4096 个 float = 16 KB。修改时须重新核算 UB 占用（见 3.1 节） */
#ifndef LAB_TILE_LENGTH
#define LAB_TILE_LENGTH (4 * 1024)
#endif
constexpr uint32_t TILE_LENGTH = static_cast<uint32_t>(LAB_TILE_LENGTH);

/* 参与计算的核数（v2/v3/v4 使用；v1 固定为 1） */
#ifndef LAB_BLOCK_DIM
#define LAB_BLOCK_DIM (8)
#endif
constexpr uint32_t BLOCK_DIM = static_cast<uint32_t>(LAB_BLOCK_DIM);

/* 计时参数：预热次数与重复次数 */
#ifndef LAB_REPEAT
#define LAB_REPEAT (50)
#endif
constexpr int32_t REPEAT = static_cast<int32_t>(LAB_REPEAT);
constexpr int32_t WARMUP = 3;

/* 对齐常量：搬运以 32 字节为单位，对 float 即 8 个元素 */
constexpr uint32_t ALIGN_ELEM = 32 / sizeof(LabDType);

/* 队列深度：TQue 模板的第二个参数，编译期常量，本实验统一取 2；
 * 实际使用几块缓冲由 InitBuffer 的第二个参数（运行时值）决定 */
constexpr uint32_t QUEUE_DEPTH = 2;

### 7.2 v1：单核基线 `KernelAddSingleCore`

本版建立三段流水的完整框架：只使用一个核，缓冲块数为 1。

算子类的成员与矢量编程范式一一对应：三个私有方法就是三个流水任务，三个 `TQue` 是它们之间的三条通道。有两点需要注意：

- `TPipe pipe` 是成员变量，每个类恰好一个。一个核函数只能初始化一个 `TPipe`。
- 位置枚举统一使用 `TPosition`。`QuePosition` 是它的别名，官方正文使用前者，本章与之保持一致。

`Init` 只做两件事：用 `SetGlobalBuffer(指针, 长度)` 把核函数收到的裸地址包装成 `GlobalTensor`，以及用 `pipe.InitBuffer(队列, 缓冲块数, 单块字节数)` 从 UB 中划分出三个队列的空间。本版是单核实现，因此不涉及 `GetBlockIdx()`。

> `InitBuffer` 的第三个参数是**字节数**，而 `DataCopy` 的第三个参数是**元素个数**。这两者容易混淆，混淆的后果是 UB 占用变为预期的 4 倍或四分之一。

`Process` 中 `CopyIn → Compute → CopyOut` 顺序调用。**在 v1 中三者确实是串行的**：缓冲块数为 1，队列里只有一块缓冲，`CopyIn` 必须等待上一轮的 `Compute` 调用 `FreeTensor` 归还缓冲后才能继续。v1 的意义正在于把「没有任何重叠」的情形量化出来，作为后续版本的参照。

三个流水任务的写法是固定的：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 流水任务 | 借缓冲 | 主体操作 | 交接 | 还缓冲 |
| --- | --- | --- | --- | --- |
| CopyIn | `AllocTensor` | `DataCopy` | `EnQue` | — |
| Compute | `AllocTensor`（输出） | `Add` | `EnQue` + `DeQue` | `FreeTensor`（输入） |
| CopyOut | — | `DataCopy` | `DeQue` | `FreeTensor` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">流水任务</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">借缓冲</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">主体操作</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">交接</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">还缓冲</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CopyIn</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>AllocTensor</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>DataCopy</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>EnQue</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Compute</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>AllocTensor</code>（输出）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Add</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>EnQue</code> + <code>DeQue</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>FreeTensor</code>（输入）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CopyOut</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>DataCopy</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>DeQue</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>FreeTensor</code></td>
</tr>
</tbody>
</table>

**每一个 `AllocTensor` 都有一个 `FreeTensor` 与之对应，每一个 `EnQue` 都有一个 `DeQue`。** 这是检查 Ascend C 算子最快的一条自检规则。

请留意 `Compute` 中 `FreeTensor` 的位置：输入缓冲在计算完成后立即归还，而不是等到 `CopyOut` 之后，这样下一轮的 `CopyIn` 才能尽早取得缓冲。思考题第 1 题将分析把它移到后面的后果。

另外，整个算子真正执行计算的只有 `AscendC::Add` 一行，它从 v1 到 v4 不会改变。**全部的性能差异都来自数据如何搬运，而非如何计算。**

In [ ]:
%%writefile -a src_add/ascendc_vector_add.asc
/* ===================== v1：单核基线 =====================
 * 新增概念：三段流水编程范式
 *   TPipe 统一管理片上内存，TQue 承担流水任务之间的数据传递与同步，
 *   核内处理逻辑划分为 CopyIn、Compute、CopyOut 三个流水任务。
 * 本版约束：仅使用一个核（blockDim = 1），缓冲块数为 1，
 *           同一时刻只有一个流水任务处于工作状态。
 */
class KernelAddSingleCore {
 public:
  __aicore__ inline KernelAddSingleCore() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z) {
    /* 把裸指针包装成 GlobalTensor：参数是（起始地址, 元素个数） */
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(x), TOTAL_LENGTH);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(y), TOTAL_LENGTH);
    zGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(z), TOTAL_LENGTH);

    /* InitBuffer 三个参数：队列对象、缓冲块数、单块缓冲的【字节数】
         * 占用 = 3 队列 x 4096 元素 x 4 字节 x 1 块 = 48 KB，占 UB(192KB) 的 25% */
    pipe.InitBuffer(inQueueX, 1, TILE_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(inQueueY, 1, TILE_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(outQueueZ, 1, TILE_LENGTH * sizeof(LabDType));
  }

  __aicore__ inline void Process() {
    const uint32_t tileNum = TOTAL_LENGTH / TILE_LENGTH;
    for (uint32_t i = 0; i < tileNum; ++i) {
      CopyIn(i);  /* 搬运单元：GM -> UB，随后 EnQue  */
      Compute();  /* 矢量单元：DeQue -> Add -> EnQue */
      CopyOut(i); /* 搬运单元：DeQue -> GM，随后 FreeTensor */
    }
  }

 private:
  __aicore__ inline void CopyIn(uint32_t progress) {
    /* 借缓冲：若队列中没有空闲块，会在此处等待下游 FreeTensor */
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.AllocTensor<LabDType>();
    AscendC::LocalTensor<LabDType> yLocal = inQueueY.AllocTensor<LabDType>();
    /* 搬数据：第三个参数是【元素个数】 */
    AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
    AscendC::DataCopy(yLocal, yGm[progress * TILE_LENGTH], TILE_LENGTH);
    /* 入队：通知下游 Compute「数据已就绪」 */
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
  }

  __aicore__ inline void Compute() {
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.DeQue<LabDType>();
    AscendC::LocalTensor<LabDType> yLocal = inQueueY.DeQue<LabDType>();
    AscendC::LocalTensor<LabDType> zLocal = outQueueZ.AllocTensor<LabDType>();

    /* 整个算子唯一执行计算的一行：一条指令完成 TILE_LENGTH 个元素的相加 */
    AscendC::Add(zLocal, xLocal, yLocal, TILE_LENGTH);

    outQueueZ.EnQue(zLocal);
    /* 输入缓冲在计算完成后立即归还，使下一轮 CopyIn 可尽早取得缓冲 */
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
  }

  __aicore__ inline void CopyOut(uint32_t progress) {
    AscendC::LocalTensor<LabDType> zLocal = outQueueZ.DeQue<LabDType>();
    AscendC::DataCopy(zGm[progress * TILE_LENGTH], zLocal, TILE_LENGTH);
    outQueueZ.FreeTensor(zLocal);
  }

  AscendC::TPipe pipe; /* 每个核函数只能有一个 */
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueX;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueY;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueueZ;
  AscendC::GlobalTensor<LabDType> xGm, yGm, zGm;
};

### 7.3 v2 与 v3：多核切分与双缓冲 `KernelAddMultiCore`

v2 相对 v1 只做一件事：**按 `block_idx` 切分数据**。

```cpp
const uint32_t blockLength = TOTAL_LENGTH / AscendC::GetBlockNum();
const uint32_t offset      = AscendC::GetBlockIdx() * blockLength;
xGm.SetGlobalBuffer(ptr + offset, blockLength);   // 偏移写入此处
```

**偏移写入 `SetGlobalBuffer` 而不是写入每次 `DataCopy`**，这样 `CopyIn` 与 `CopyOut` 中的下标表达式与 v1 完全一致，「各核处理各自分片」这一语义被收敛在 `Init` 一处。版本之间的差异越集中，越便于对照阅读。

v3 相对 v2 也只做一件事：**缓冲块数由 1 改为 2**。既然差异只有一个数值，就把它作为 `Init` 的参数，不再复制一份算子类：

```cpp
op.Init(x, y, z, /*bufferNum=*/1);   // v2
op.Init(x, y, z, /*bufferNum=*/2);   // v3
```

`InitBuffer` 的第二个参数本来就接受运行时值，因此这样写完全合法，也更清楚地表明二者的差异所在。

本版假定 `TOTAL_LENGTH` 可被核数与分块长度整除。代码中用 `static_assert` 把这一假定变为**编译期错误**，而不是运行时的静默错算——**隐含假设应当写入代码，而不是只写在注释里**。非整除的一般情形由 v4 处理。

In [ ]:
%%writefile -a src_add/ascendc_vector_add.asc
/* ===================== v2 / v3：多核 + 可变缓冲块数 =====================
 * v2 = Init(..., bufferNum = 1)：新增 SPMD 多核数据切分
 * v3 = Init(..., bufferNum = 2)：新增双缓冲（与 v2 唯一的差别就是这个数值）
 * 本版假定 TOTAL_LENGTH 可被 BLOCK_DIM 与 TILE_LENGTH 整除，见下方 static_assert。
 */
static_assert(TOTAL_LENGTH % (LAB_BLOCK_DIM * LAB_TILE_LENGTH) == 0,
              "v2/v3 假定 TOTAL_LENGTH 可被 BLOCK_DIM*TILE_LENGTH 整除；"
              "非整除的一般情形请使用 v4");

class KernelAddMultiCore {
 public:
  __aicore__ inline KernelAddMultiCore() {}

  /* bufferNum = 1 即单缓冲（v2），= 2 即双缓冲（v3） */
  __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z,
                              uint32_t bufferNum) {
    /* ---------- 第一级：多核切分 ---------- */
    blockLength_ = TOTAL_LENGTH / AscendC::GetBlockNum();
    const uint32_t offset = AscendC::GetBlockIdx() * blockLength_;

    /* 偏移直接写入 SetGlobalBuffer，使核内其余代码与 v1 完全一致 */
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(x) + offset,
                        blockLength_);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(y) + offset,
                        blockLength_);
    zGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(z) + offset,
                        blockLength_);

    /* 第二个参数是【缓冲块数】，接受运行时值：1 = 单缓冲，2 = 双缓冲 */
    pipe.InitBuffer(inQueueX, bufferNum, TILE_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(inQueueY, bufferNum, TILE_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(outQueueZ, bufferNum, TILE_LENGTH * sizeof(LabDType));
  }

  __aicore__ inline void Process() {
    /* ---------- 第二级：核内分块 ---------- */
    const uint32_t tileNum = blockLength_ / TILE_LENGTH;
    for (uint32_t i = 0; i < tileNum; ++i) {
      CopyIn(i);
      Compute();
      CopyOut(i);
    }
  }

 private:
  /* 以下三个函数与 v1 逐字相同——这正是把偏移写入 SetGlobalBuffer 的好处 */
  __aicore__ inline void CopyIn(uint32_t progress) {
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.AllocTensor<LabDType>();
    AscendC::LocalTensor<LabDType> yLocal = inQueueY.AllocTensor<LabDType>();
    AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
    AscendC::DataCopy(yLocal, yGm[progress * TILE_LENGTH], TILE_LENGTH);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
  }

  __aicore__ inline void Compute() {
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.DeQue<LabDType>();
    AscendC::LocalTensor<LabDType> yLocal = inQueueY.DeQue<LabDType>();
    AscendC::LocalTensor<LabDType> zLocal = outQueueZ.AllocTensor<LabDType>();
    AscendC::Add(zLocal, xLocal, yLocal, TILE_LENGTH);
    outQueueZ.EnQue(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
  }

  __aicore__ inline void CopyOut(uint32_t progress) {
    AscendC::LocalTensor<LabDType> zLocal = outQueueZ.DeQue<LabDType>();
    AscendC::DataCopy(zGm[progress * TILE_LENGTH], zLocal, TILE_LENGTH);
    outQueueZ.FreeTensor(zLocal);
  }

  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueX;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueY;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueueZ;
  AscendC::GlobalTensor<LabDType> xGm, yGm, zGm;
  uint32_t blockLength_ = 0;
};

### 7.4 v4：变长输入与尾核尾块 `KernelAddVarLength`

前三个版本都假定长度可被整除，真实算子必须处理任意长度。v4 做两处改动。

**其一，元素总数改为运行时参数**，由 Host 侧作为核函数的第四个参数传入。同一份核函数二进制因而可以处理不同长度的输入。

需要说明本实验与官方推荐做法的关系。CANN 主推的方式是**由 Host 侧的 Tiling 函数计算全部切分参数**，打包进 `TilingData` 结构体后随核函数下发，Kernel 侧再取出使用；官方的算子工程正是围绕这一机制组织的。本实验为了把注意力集中在核内的编程范式上，前三个版本使用编译期常量，v4 只传入一个长度参数——这是同一机制的最小形态。当参数增多到需要成组传递时，它就演化为 `TilingData`，完整流程见实验七。

**其二，核内显式区分三种情形**：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 步骤 | 做法 | 目的 |
| --- | --- | --- |
| 每核基础长度 | `(totalLength / blockNum) / 8 * 8`，**向下**对齐 | 保证各核起始地址为 32 字节对齐 |
| **核间余量**（简化的尾核处理） | 末核长度 = `totalLength - offset`，承担全部余量 | 不遗漏任何元素。官方的尾核方案是把余量均摊给若干个核，见 §3.2 |
| **尾块** | 核内先处理 `tileNum` 个完整块，再以实际长度处理一次 | 处理不足一个完整分块的部分 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">步骤</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">做法</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">目的</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每核基础长度</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>(totalLength / blockNum) / 8 * 8</code>，<strong>向下</strong>对齐</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">保证各核起始地址为 32 字节对齐</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>核间余量</strong>（简化的尾核处理）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">末核长度 = <code>totalLength - offset</code>，承担全部余量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不遗漏任何元素。官方的尾核方案是把余量均摊给若干个核，见 §3.2</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>尾块</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核内先处理 <code>tileNum</code> 个完整块，再以实际长度处理一次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">处理不足一个完整分块的部分</td>
</tr>
</tbody>
</table>

本版要求元素总数为 8 的整数倍。在此前提下，末核长度与尾块长度必然也是 8 的整数倍，全部搬运均满足对齐要求。元素总数不是 8 的整数倍的一般情形，须借助带填充能力的搬运接口 `DataCopyPad`（配合 `DataCopyExtParams` 与 `DataCopyPadExtParams`）处理，见思考题第 5 题。

三个流水任务的签名由 `(progress)` 变为 `(offset, length)`，这是 v4 相对 v3 最容易被忽略、却最关键的一处改动。

> **一处边界情形**：当 `totalLength` 小于 `blockNum × 8` 时，`perCore` 会算成 0，前若干个核的 `blockLength_` 为 0、无数据可处理，末核承担全部数据。代码对此做了保护，但保护的位置需要注意：`InitBuffer` 应当无条件执行（`TPipe` 随对象构造，不应出现一块缓冲都不初始化的情形），只跳过 `SetGlobalBuffer` 与处理循环。

In [ ]:
%%writefile -a src_add/ascendc_vector_add.asc
/* ===================== v4：变长输入、尾核与尾块 =====================
 * 新增概念：非整除长度下的正确性
 * 约束：元素总数须为 ALIGN_ELEM（float 下为 8 个元素，即 32 字节）的整数倍。
 */
class KernelAddVarLength {
 public:
  __aicore__ inline KernelAddVarLength() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z,
                              uint32_t totalLength) {
    const uint32_t blockNum = AscendC::GetBlockNum();
    const uint32_t blockIdx = AscendC::GetBlockIdx();

    /* 每核基础长度向下对齐到 ALIGN_ELEM，保证各核起始地址满足 32B 对齐 */
    const uint32_t perCore = (totalLength / blockNum) / ALIGN_ELEM * ALIGN_ELEM;
    const uint32_t offset = blockIdx * perCore;

    /* 尾核：末核承担因向下对齐而剩余的元素 */
    blockLength_ =
        (blockIdx == blockNum - 1) ? (totalLength - offset) : perCore;

    /* InitBuffer 无条件执行：TPipe 随对象构造，不应出现「一块都不初始化」的情形 */
    pipe.InitBuffer(inQueueX, 2, TILE_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(inQueueY, 2, TILE_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(outQueueZ, 2, TILE_LENGTH * sizeof(LabDType));

    /* 极小输入时前若干核可能分不到数据，此时不绑定 GM，Process 直接返回 */
    if (blockLength_ == 0) {
      return;
    }

    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(x) + offset,
                        blockLength_);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(y) + offset,
                        blockLength_);
    zGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(z) + offset,
                        blockLength_);
  }

  __aicore__ inline void Process() {
    if (blockLength_ == 0) {
      return;
    }

    const uint32_t tileNum = blockLength_ / TILE_LENGTH; /* 完整分块数 */
    const uint32_t tailLen = blockLength_ % TILE_LENGTH; /* 尾块长度   */

    for (uint32_t i = 0; i < tileNum; ++i) {
      const uint32_t off = i * TILE_LENGTH;
      CopyIn(off, TILE_LENGTH);
      Compute(TILE_LENGTH);
      CopyOut(off, TILE_LENGTH);
    }

    /* 尾块：长度小于 TILE_LENGTH，但仍为 ALIGN_ELEM 的整数倍 */
    if (tailLen > 0) {
      const uint32_t off = tileNum * TILE_LENGTH;
      CopyIn(off, tailLen);
      Compute(tailLen);
      CopyOut(off, tailLen);
    }
  }

 private:
  /* 三个流水任务的签名由 (progress) 变为 (offset, length) */
  __aicore__ inline void CopyIn(uint32_t offset, uint32_t length) {
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.AllocTensor<LabDType>();
    AscendC::LocalTensor<LabDType> yLocal = inQueueY.AllocTensor<LabDType>();
    AscendC::DataCopy(xLocal, xGm[offset], length);
    AscendC::DataCopy(yLocal, yGm[offset], length);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
  }

  __aicore__ inline void Compute(uint32_t length) {
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.DeQue<LabDType>();
    AscendC::LocalTensor<LabDType> yLocal = inQueueY.DeQue<LabDType>();
    AscendC::LocalTensor<LabDType> zLocal = outQueueZ.AllocTensor<LabDType>();
    AscendC::Add(zLocal, xLocal, yLocal, length);
    outQueueZ.EnQue(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
  }

  __aicore__ inline void CopyOut(uint32_t offset, uint32_t length) {
    AscendC::LocalTensor<LabDType> zLocal = outQueueZ.DeQue<LabDType>();
    AscendC::DataCopy(zGm[offset], zLocal, length);
    outQueueZ.FreeTensor(zLocal);
  }

  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueX;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueY;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueueZ;
  AscendC::GlobalTensor<LabDType> xGm, yGm, zGm;
  uint32_t blockLength_ = 0;
};

### 7.5 四个核函数入口

核函数只做三件事：实例化算子类、调用 `Init`、调用 `Process`。

每个核函数的第一行都是 `KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);`，含义是本核函数只使用矢量核（AIV）。

**本实验必须写这一行，理由是明确的**：官方规定，在**同一个编译单元中存在多个核函数**时，暂不支持自动推导 Kernel 类型，需要开发者手动设置。本实验的四个核函数 `add_v1` 至 `add_v4` 写在同一个 `.asc` 文件中，正好落在这一条上。（另外两种不推导的情形是纯标量算子，以及 Atlas 350 加速卡与 Atlas 推理系列产品。）

更进一步，即使在可以自动推导的场合，显式声明也有必要。`blockDim` 是**逻辑核**的概念，它对应多少物理资源取决于 Kernel 类型：在分离模式下，纯矢量算子的 `blockDim` 就是启动的矢量核数量；而矢量与矩阵的**融合**算子按组合启动，一个组合为 2 个矢量核加 1 个矩阵核，此时 `blockDim` 表示的是组合数。**同一个 `blockDim = 8`，在两种 Kernel 类型下对应的矢量核数量并不相同**，§12.1 核数扫描的横轴含义也随之改变。

这与实验一 §4 讨论的是同一个问题。**应当把「只使用矢量核」这一约定写入代码，而不是交由编译器推断。**

In [ ]:
%%writefile -a src_add/ascendc_vector_add.asc
/* ===================== 核函数入口 ===================== */

__global__ __aicore__ void add_v1(GM_ADDR x, GM_ADDR y, GM_ADDR z) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY); /* 声明为纯矢量内核 */
  KernelAddSingleCore op;
  op.Init(x, y, z);
  op.Process();
}

__global__ __aicore__ void add_v2(GM_ADDR x, GM_ADDR y, GM_ADDR z) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelAddMultiCore op;
  op.Init(x, y, z, /*bufferNum=*/1); /* 单缓冲 */
  op.Process();
}

__global__ __aicore__ void add_v3(GM_ADDR x, GM_ADDR y, GM_ADDR z) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelAddMultiCore op;
  op.Init(x, y, z, /*bufferNum=*/2); /* 双缓冲：与 v2 唯一的差别 */
  op.Process();
}

__global__ __aicore__ void add_v4(GM_ADDR x, GM_ADDR y, GM_ADDR z,
                                  uint32_t totalLength) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelAddVarLength op;
  op.Init(x, y, z, totalLength);
  op.Process();
}

## 8. Host 侧实现

Device 侧到此结束。Host 侧的代码分两部分写入：先是一组基础设施，然后是主程序。

### 8.1 基础设施：错误检查、计时、数据生成、CPU 基准与结果校验

这一段包含五组内容，它们互相配合完成「造数据 — 算参考值 — 计时 — 校验」的完整闭环：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 内容 | 说明 |
| --- | --- |
| `ACL_CHECK` | 与实验一相同，任何 ACL 接口返回非 0 即报告位置并退出 |
| 计时与设备管理 | `GetTimeMs` 用 `clock_gettime(CLOCK_MONOTONIC)` 取毫秒时刻；`NpuInit` / `NpuFinalize` 中的设备初始化开销在百毫秒量级，因此放在测量循环之外，全程只执行一次 |
| `GenerateInput` | 用固定种子的线性同余发生器在程序内部生成输入，不读写任何文件，也不依赖 `<random>`（原因见下） |
| `RunOnCpu` | CPU 单线程基准。其结果即校验用的参考值，因此不会被编译器消除（见 §5.2 第 5 点） |
| `Verify` | 统一采用**相对误差**校验。本算子的单精度加法在两侧通常逐位一致，但该口径对后续含归约或高阶函数的算子同样适用 |
| `TIME_KERNEL` | 对应 §5.1 的 `npu_kernel_ms`：先预热，再重复下发并逐次同步后取平均 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_CHECK</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与实验一相同，任何 ACL 接口返回非 0 即报告位置并退出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">计时与设备管理</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GetTimeMs</code> 用 <code>clock_gettime(CLOCK_MONOTONIC)</code> 取毫秒时刻；<code>NpuInit</code> / <code>NpuFinalize</code> 中的设备初始化开销在百毫秒量级，因此放在测量循环之外，全程只执行一次</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GenerateInput</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">用固定种子的线性同余发生器在程序内部生成输入，不读写任何文件，也不依赖 <code>&lt;random&gt;</code>（原因见下）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>RunOnCpu</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CPU 单线程基准。其结果即校验用的参考值，因此不会被编译器消除（见 §5.2 第 5 点）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Verify</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">统一采用<strong>相对误差</strong>校验。本算子的单精度加法在两侧通常逐位一致，但该口径对后续含归约或高阶函数的算子同样适用</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TIME_KERNEL</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">对应 §5.1 的 <code>npu_kernel_ms</code>：先预热，再重复下发并逐次同步后取平均</td>
</tr>
</tbody>
</table>

In [ ]:
%%writefile -a src_add/ascendc_vector_add.asc
/* ============================================================
 *                       Host 侧代码
 * ============================================================ */

#define ACL_CHECK(expr)                                                       \
  do {                                                                        \
    aclError _ret = (expr);                                                   \
    if (_ret != ACL_SUCCESS) {                                                \
      std::printf("[ACL ERROR] %s:%d  %s  returned %d\n", __FILE__, __LINE__, \
                  #expr, static_cast<int>(_ret));                             \
      std::exit(EXIT_FAILURE);                                                \
    }                                                                         \
  } while (0)

/* 取当前时刻，单位毫秒。
 * 使用 CLOCK_MONOTONIC 单调时钟：它自系统启动起单调递增，不受系统时间调整
 * （NTP 校时、手动改时间）影响，是测量时间间隔的标准做法。 */
static inline double GetTimeMs() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1000000.0;
}

static int32_t g_deviceId = 0;
static aclrtStream g_stream = nullptr;

/* 设备初始化与销毁开销在百毫秒量级，全程只执行一次，不放入计时循环 */
static void NpuInit() {
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(g_deviceId));
  ACL_CHECK(aclrtCreateStream(&g_stream));
}

static void NpuFinalize() {
  ACL_CHECK(aclrtDestroyStream(g_stream));
  ACL_CHECK(aclrtResetDevice(g_deviceId));
  ACL_CHECK(aclFinalize());
}

/* ---------- 数据生成：固定种子，CPU 与 NPU 使用同一份输入 ----------
 * 采用线性同余发生器（LCG），只用整数乘加与一次浮点乘法，
 * 不依赖 <random>，也不依赖任何数学库函数。原因见 8.1 节的说明。
 */
static inline float LcgNextFloat(uint32_t &state) {
  state = state * 1664525u + 1013904223u; /* Numerical Recipes 推荐参数 */
  /* 取高 24 位（避免低位周期短），线性映射到 [-1, 1) */
  return static_cast<float>(state >> 8) * (2.0f / 16777216.0f) - 1.0f;
}

static void GenerateInput(std::vector<LabDType> &x, std::vector<LabDType> &y,
                          uint32_t seed = 2026u) {
  uint32_t s = seed;
  for (size_t i = 0; i < x.size(); ++i) {
    x[i] = static_cast<LabDType>(LcgNextFloat(s));
    y[i] = static_cast<LabDType>(LcgNextFloat(s));
  }
}

/* ---------- CPU 单线程基准；其结果同时作为校验用的参考值 ---------- */
static double RunOnCpu(const std::vector<LabDType> &x,
                       const std::vector<LabDType> &y, std::vector<LabDType> &z,
                       int warmup, int repeat) {
  const size_t n = x.size();

  for (int r = 0; r < warmup; ++r) /* 预热：消除缺页与冷 cache 的影响 */
    for (size_t i = 0; i < n; ++i) z[i] = x[i] + y[i];

  const double t0 = GetTimeMs();
  for (int r = 0; r < repeat; ++r) /* 多次重复取平均 */
    for (size_t i = 0; i < n; ++i) z[i] = x[i] + y[i];
  return (GetTimeMs() - t0) / repeat;
}

/* ---------- 相对误差校验 ----------
 * 逐元素单精度加法在两侧均按 IEEE 754 执行，结果通常逐位一致；但一旦算子中
 * 出现融合乘加、归约次序不同或高阶数学函数，两侧的舍入就会产生差异。
 * 因此本章统一采用相对误差校验，而不依赖位级比对。 */
static bool Verify(const char *ver, uint32_t n,
                   const std::vector<LabDType> &out,
                   const std::vector<LabDType> &golden, double eps = 1e-5) {
  double maxRelErr = 0.0;
  size_t badIdx = 0, badCount = 0;
  for (size_t i = 0; i < golden.size(); ++i) {
    const double g = static_cast<double>(golden[i]);
    const double o = static_cast<double>(out[i]);
    const double rel = std::fabs(o - g) / std::max(1e-6, std::fabs(g));
    if (rel > maxRelErr) {
      maxRelErr = rel;
      badIdx = i;
    }
    if (rel > eps) {
      ++badCount;
    }
  }
  const bool ok = (badCount == 0);
  std::printf(
      "[VERIFY] ver=%s n=%u max_rel_err=%.3e at=%zu bad=%zu result=%s\n", ver,
      n, maxRelErr, badIdx, badCount, ok ? "PASS" : "FAIL");
  return ok;
}

/* ---------- 计时宏：核函数耗时（不含主机与设备之间的数据搬运） ---------- */
#define TIME_KERNEL(LAUNCH, OUT_MS)                                               \
  do {                                                                            \
    for (int _w = 0; _w < WARMUP; ++_w) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream));                                \
    }                                                                             \
    const double _t0 = GetTimeMs();                                               \
    for (int _r = 0; _r < REPEAT; ++_r) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream)); /* 先同步再停止计时 */ \
    }                                                                             \
    (OUT_MS) = (GetTimeMs() - _t0) / REPEAT;                                      \
  } while (0)

/* ---------- 打印一行可被程序解析的性能记录 ----------
 * 加速比与其分子、分母一并输出：脱离基准耗时的加速比无法解读，原因见 13 节第 5 条。 */
static void ReportPerf(const char *ver, uint32_t n, uint32_t blockDim,
                       uint32_t bufNum, double cpuMs, double kernelMs) {
  const double ubKB = 3.0 * TILE_LENGTH * sizeof(LabDType) * bufNum / 1024.0;
  std::printf(
      "[PERF]   ver=%s n=%u blockDim=%u bufNum=%u ub_kb=%.1f "
      "cpu_ms=%.4f kernel_ms=%.4f sp_kernel=%.4f\n",
      ver, n, blockDim, bufNum, ubKB, cpuMs, kernelMs, cpuMs / kernelMs);
}

### 8.2 主程序

流程为：生成数据 → 运行 CPU 基准（同时得到参考值）→ 申请显存并搬入 → 逐个版本计时、回读、校验、打印 → 释放资源。

**每个版本运行前都把 `zDev` 清零。** 这一步看似多余，实则必要：如果某个版本因切分错误漏算了一部分数据，不清零时它会读到上一个版本留下的正确结果，校验仍然通过，错误被完全掩盖。这类陈旧输出是性能实验中较为隐蔽的一类问题。

命令行支持一个可选参数 `N`：不带参数时运行四个版本的对照；带参数时只运行 v4，用于 §10 的变长验证与 §12.3 的规模扫描，两者都不需要重新编译。

In [ ]:
%%writefile -a src_add/ascendc_vector_add.asc
int32_t main(int argc, char *argv[]) {
  /* ---------- 解析命令行 ---------- */
  bool onlyV4 = false;
  uint32_t n = TOTAL_LENGTH;
  if (argc > 1) {
    onlyV4 = true;
    n = static_cast<uint32_t>(std::strtoul(argv[1], nullptr, 10));
    if (n == 0 || n % ALIGN_ELEM != 0) {
      std::printf("[FATAL] N 必须是 %u 的整数倍（32 字节对齐要求）\n",
                  ALIGN_ELEM);
      return 1;
    }
  }

  NpuInit();

  /* ---------- 生成数据 ---------- */
  std::vector<LabDType> x(n), y(n), golden(n), out(n);
  GenerateInput(x, y);

  /* ---------- CPU 基准，其结果即参考值 ---------- */
  const double cpuMs =
      RunOnCpu(x, y, golden, WARMUP, (n > (4u << 20)) ? 5 : 20);

  /* ---------- 申请显存并搬入输入 ---------- */
  const size_t bytes = static_cast<size_t>(n) * sizeof(LabDType);
  void *hostX = x.data(), *hostY = y.data();
  uint8_t *xDev = nullptr, *yDev = nullptr, *zDev = nullptr, *zHost = nullptr;
  ACL_CHECK(aclrtMalloc((void **)&xDev, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&yDev, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&zDev, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMallocHost((void **)&zHost, bytes));
  ACL_CHECK(aclrtMemcpy(xDev, bytes, hostX, bytes, ACL_MEMCPY_HOST_TO_DEVICE));
  ACL_CHECK(aclrtMemcpy(yDev, bytes, hostY, bytes, ACL_MEMCPY_HOST_TO_DEVICE));

  std::printf("N=%u  TILE_LENGTH=%u  BLOCK_DIM=%u  REPEAT=%d  cpu_ms=%.4f\n", n,
              TILE_LENGTH, BLOCK_DIM, REPEAT, cpuMs);

  bool allPass = true;
  double kMs = 0.0;

/* 每个版本运行前清零 zDev，避免读到上一个版本留下的陈旧输出 */
#define RESET_Z() ACL_CHECK(aclrtMemset(zDev, bytes, 0, bytes))
#define FETCH_Z()                                                           \
  do {                                                                      \
    ACL_CHECK(                                                              \
        aclrtMemcpy(zHost, bytes, zDev, bytes, ACL_MEMCPY_DEVICE_TO_HOST)); \
    out.assign(reinterpret_cast<LabDType *>(zHost),                         \
               reinterpret_cast<LabDType *>(zHost) + n);                    \
  } while (0)

  if (!onlyV4) {
    /* ---------------- v1：单核基线 ---------------- */
    RESET_Z();
    TIME_KERNEL((add_v1<<<1, nullptr, g_stream>>>(xDev, yDev, zDev)), kMs);
    FETCH_Z();
    allPass &= Verify("v1", n, out, golden);
    ReportPerf("v1", n, 1, 1, cpuMs, kMs);

    /* ---------------- v2：多核 ---------------- */
    RESET_Z();
    TIME_KERNEL((add_v2<<<BLOCK_DIM, nullptr, g_stream>>>(xDev, yDev, zDev)),
                kMs);
    FETCH_Z();
    allPass &= Verify("v2", n, out, golden);
    ReportPerf("v2", n, BLOCK_DIM, 1, cpuMs, kMs);

    /* ---------------- v3：多核 + 双缓冲 ---------------- */
    RESET_Z();
    TIME_KERNEL((add_v3<<<BLOCK_DIM, nullptr, g_stream>>>(xDev, yDev, zDev)),
                kMs);
    FETCH_Z();
    allPass &= Verify("v3", n, out, golden);
    ReportPerf("v3", n, BLOCK_DIM, 2, cpuMs, kMs);
  }

  /* ---------------- v4：变长 + 尾核 + 尾块 ---------------- */
  RESET_Z();
  TIME_KERNEL((add_v4<<<BLOCK_DIM, nullptr, g_stream>>>(xDev, yDev, zDev, n)),
              kMs);
  FETCH_Z();
  allPass &= Verify("v4", n, out, golden);
  ReportPerf("v4", n, BLOCK_DIM, 2, cpuMs, kMs);

  /* ---------- 释放资源（与申请严格成对，顺序相反） ---------- */
  ACL_CHECK(aclrtFreeHost(zHost));
  ACL_CHECK(aclrtFree(zDev));
  ACL_CHECK(aclrtFree(yDev));
  ACL_CHECK(aclrtFree(xDev));
  NpuFinalize();

  std::printf(allPass ? "[SUCCESS] 全部版本校验通过。\n"
                      : "[FAILED] 存在校验未通过的版本！\n");
  return allPass ? 0 : 1;
}

## 9. 编译与运行

一条 `bisheng` 命令即可完成编译：Host 侧 C++ 与 Device 侧四个核函数一起编译，产出单个可执行文件。

`-O2` 不能省略——CPU 基准与 NPU 代码在同一次编译中生成，使用 `-O0` 会人为放大基准实现的耗时，使加速比失真。

In [ ]:
import subprocess

ARCH = "dav-2201"  # ← 若设备不是 Atlas A2/A3，请按实验一 §7.2 的表修改
SRC = "src_add/ascendc_vector_add.asc"
EXE = "src_add/ascendc_vector_add"

# bisheng [算子源文件] --npu-arch=[NPU架构版本号] -O2 -o [输出产物名称]
cmd = ["bisheng", SRC, "--npu-arch=" + ARCH, "-O2", "-o", EXE]
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)

一次运行即可输出四个版本的校验结果与性能记录。

In [ ]:
import subprocess


def run_demo(args=(), timeout=900):
    proc = subprocess.run(
        ["./src_add/ascendc_vector_add"] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if proc.returncode != 0 and not proc.stdout:
        print("返回码", proc.returncode)
        print(proc.stderr)
    return proc.stdout


out_main = run_demo()
print(out_main)

### 9.1 解析输出

下面两个函数把 `[PERF]` 与 `[VERIFY]` 记录行解析为字典，后续所有表格与图表都基于它们。

In [ ]:
import re


def parse_perf(text):
    # 把所有 [PERF] 行解析为 dict 列表
    rows = []
    for line in text.splitlines():
        if line.startswith("[PERF]"):
            d = {}
            for kv in line.split()[1:]:
                k, v = kv.split("=", 1)
                d[k] = v if k == "ver" else float(v)
            rows.append(d)
    return rows


def parse_verify(text):
    return {
        m.group(1): m.group(2)
        for m in re.finditer(r"\[VERIFY\] ver=(\S+).*?result=(\S+)", text)
    }


rows_main = parse_perf(out_main)
chk_main = parse_verify(out_main)
base_v1 = rows_main[0]["kernel_ms"] if rows_main else 1.0

hdr = (
    "版本",
    "核数",
    "buf",
    "UB(KB)",
    "CPU(ms)",
    "kernel(ms)",
    "vs CPU",
    "vs v1",
    "校验",
)
print("%-5s %5s %4s %8s %10s %11s %9s %8s %6s" % hdr)
for r in rows_main:
    print(
        "%-5s %5d %4d %8.1f %10.4f %11.4f %8.2fx %7.2fx %6s"
        % (
            r["ver"],
            r["blockDim"],
            r["bufNum"],
            r["ub_kb"],
            r["cpu_ms"],
            r["kernel_ms"],
            r["sp_kernel"],
            base_v1 / r["kernel_ms"],
            chk_main.get(r["ver"], "?"),
        )
    )

## 10. v4 的变长验证

前面四个版本使用的都是 N = 2^21，可以被核数和分块长度整除。下面换一个会触发尾核与尾块的长度。

取 N_odd = 2 097 152 + 1000 = 2 098 152。它满足：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 条件 | 计算 | 结论 |
| --- | --- | --- |
| 是 `ALIGN_ELEM = 8` 的整数倍？ | 2098152 / 8 = 262269 | 满足对齐约束 |
| 每核基础长度是 8 的整数倍？ | 2098152 / 8 = 262269，**262269 不是 8 的倍数** | 否 → 向下对齐到 262264，**触发尾核** |
| 末核长度能被分块长度整除？ | 末核 = 2098152 − 7 × 262264 = 262304；262304 mod 4096 = 160 | 否 → **触发尾块**（160 个元素） |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">条件</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">计算</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">结论</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">是 <code>ALIGN_ELEM = 8</code> 的整数倍？</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2098152 / 8 = 262269</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">满足对齐约束</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每核基础长度是 8 的整数倍？</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2098152 / 8 = 262269，<strong>262269 不是 8 的倍数</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否 → 向下对齐到 262264，<strong>触发尾核</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">末核长度能被分块长度整除？</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">末核 = 2098152 − 7 × 262264 = 262304；262304 mod 4096 = 160</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否 → <strong>触发尾块</strong>（160 个元素）</td>
</tr>
</tbody>
</table>

> **注意这里的两个「8」含义不同**：一个是核数 `BLOCK_DIM = 8`，一个是对齐单位 `ALIGN_ELEM = 8`。N_odd **可以**被核数 8 整除，真正触发尾核的原因是每核基础长度 262269 不是对齐单位 8 的倍数。

因为长度是运行时参数，不需要重新编译，直接把它作为命令行参数传入即可。

In [ ]:
N_ODD = 2097152 + 1000

assert N_ODD % 8 == 0, "元素总数须为 ALIGN_ELEM(=8) 的整数倍"
assert (N_ODD // 8) % 8 != 0, "每核基础长度应当不是 8 的倍数，才会触发尾核"

per_core = (N_ODD // 8) // 8 * 8
last_core = N_ODD - 7 * per_core
assert last_core % 4096 != 0, "末核长度应当不能被 TILE_LENGTH 整除，才会触发尾块"

print("每核基础长度 :", per_core)
print(
    "末核长度     :",
    last_core,
    "→ 完整块 %d 个 + 尾块 %d 个元素" % (last_core // 4096, last_core % 4096),
)
print()

out_odd = run_demo([N_ODD])
print(out_odd)

校验通过说明尾核与尾块都处理正确。动手练习第 2 题将用 v3 运行同一个长度，观察它的失败方式，并解释超差的起点为什么出现在预期之前。

## 11. 结果可视化

下面绘制两张加速比图，它们回答的是两个不同的问题：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 图 | 基线 | 回答的问题 |
| --- | --- | --- |
| 图一 | **CPU 单线程** | 使用 NPU 的整体收益 |
| 图二 | **v1（单核 NPU）** | 每一项优化各自的贡献 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">图</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">基线</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">回答的问题</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图一</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>CPU 单线程</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">使用 NPU 的整体收益</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图二</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1（单核 NPU）</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每一项优化各自的贡献</td>
</tr>
</tbody>
</table>

图一同时给出 kernel-only 与 end-to-end 两组数据，并画一条 y = 1 的基准线作为参照。请特别注意 end-to-end 一组相对该基准线的位置，§13 ④ 将对它作出解释。

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

C_KERNEL, C_ALT, C_BASE, C_OPT = "#3B6FE0", "#E07A3B", "#9AA5B1", "#2E9E6B"

vers = [r["ver"] for r in rows_main]
spk = [r["sp_kernel"] for r in rows_main]
xpos = np.arange(len(vers))

fig, ax = plt.subplots(figsize=(7.6, 4.3), dpi=120)
ax.bar(xpos, spk, 0.5, color=C_KERNEL, label="NPU kernel only")
ax.axhline(1.0, color="#888888", lw=1.0, ls=":")
ax.text(-0.45, 1.06, "baseline: CPU 1 thread = 1.0x", fontsize=9, color="#777777")
for i, a in enumerate(spk):
    ax.text(i, a, "%.1fx" % a, ha="center", va="bottom", fontsize=8)
ax.set_xticks(xpos)
ax.set_xticklabels(vers)
ax.set_yscale("log")
ax.set_ylabel("Speedup over CPU baseline")
ax.set_title("Lab 2: Vector Add - NPU kernel speedup vs single-thread CPU")
ax.grid(axis="y", alpha=0.3, which="both")
ax.legend(frameon=False, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
sp_v1 = [base_v1 / r["kernel_ms"] for r in rows_main]
colors = [C_BASE] + [C_OPT] * (len(vers) - 1)

fig, ax = plt.subplots(figsize=(6.8, 4.0), dpi=120)
ax.bar(vers, sp_v1, 0.55, color=colors)
for i, v in enumerate(sp_v1):
    ax.text(i, v, "%.2fx" % v, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Speedup vs. v1 (single core, kernel only)")
ax.set_title("Lab 2: contribution of each optimization step")
ax.grid(axis="y", alpha=0.3)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 12. 参数扫描

上述四个版本回答的是「是否采用某项优化」。本节考察两个连续参数的取值影响，方法与第五章 OpenMP 实验中的线程数扫描一致。

修改参数不需要修改源码：直接在 `bisheng` 命令后追加 `-D` 即可，每次编译只需几秒。

### 12.1 核数扫描

固定分块长度，改变参与计算的核数。加速比会随核数上升，但通常低于线性，且偏离线性的程度随核数增大。偏离的原因有两个层次：核启动与调度的固定开销所占比重上升，以及 Global Memory 带宽逼近上限。§13 ⑥ 将给出区分二者的判别方法。

In [ ]:
import subprocess


def build_and_run(defines=None, args=(), tag="scan"):
    # 按给定的 -D 宏重新编译并运行，返回 (stdout, 是否成功)
    exe = "src_add/ascendc_vector_add_%s" % tag
    cmd = [
        "bisheng",
        "src_add/ascendc_vector_add.asc",
        "--npu-arch=" + ARCH,
        "-O2",
        "-o",
        exe,
    ]
    for k, v in (defines or {}).items():
        cmd.append("-D%s=%s" % (k, v))
    b = subprocess.run(cmd, capture_output=True, text=True)
    if b.returncode != 0:
        return (b.stdout + b.stderr), False
    r = subprocess.run(
        ["./" + exe] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=900,
    )
    return r.stdout, ("[PERF]" in r.stdout)


block_dims = [1, 2, 4, 8, 16, 32]
scan_core = []
for bd in block_dims:
    txt, ok = build_and_run({"LAB_BLOCK_DIM": bd}, tag="bd%d" % bd)
    if not ok:
        print("❌ blockDim=%-3d 编译或运行失败（可能超出设备可用核数）" % bd)
        continue
    r = [p for p in parse_perf(txt) if p["ver"] == "v3"]
    if r:
        scan_core.append(
            (bd, r[0]["kernel_ms"], r[0]["sp_kernel"], parse_verify(txt).get("v3"))
        )
        print(
            "blockDim=%-3d kernel_ms=%.4f  vs CPU=%.2fx  check=%s"
            % (bd, r[0]["kernel_ms"], r[0]["sp_kernel"], parse_verify(txt).get("v3"))
        )

In [ ]:
if scan_core:
    bd = [s[0] for s in scan_core]
    spc = [s[2] for s in scan_core]
    base = scan_core[0][1]
    spv1 = [base / s[1] for s in scan_core]

    fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=120)
    ax.plot(bd, spv1, marker="o", lw=2, color=C_KERNEL, label="speedup vs. blockDim=1")
    ax.plot(bd, bd, lw=1, ls="--", color="#bbbbbb", label="ideal linear")
    ax2 = ax.twinx()
    ax2.plot(bd, spc, marker="s", lw=2, ls=":", color=C_ALT, label="speedup vs. CPU")
    ax2.set_ylabel("Speedup vs. CPU", color=C_ALT)
    ax.set_xscale("log", base=2)
    ax.set_xticks(bd)
    ax.set_xticklabels(bd)
    ax.set_xlabel("blockDim (number of vector cores)")
    ax.set_ylabel("Speedup vs. single core")
    ax.set_title("Lab 2: core-count scaling (v3)")
    ax.grid(alpha=0.3)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, frameon=False, loc="upper left")
    ax.spines["top"].set_visible(False)
    plt.tight_layout()
    plt.show()

### 12.2 分块长度扫描

固定核数，改变单次搬运的元素数。这一参数存在两侧的制约：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 分块长度 | 不利因素 |
| --- | --- |
| 过小 | 单次搬运的有效载荷不足，搬运指令的固定开销占比上升；流水级数增多 |
| 过大 | **UB 占用增加，可能超出容量**；流水的启动与排空阶段变长 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">分块长度</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">不利因素</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">过小</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单次搬运的有效载荷不足，搬运指令的固定开销占比上升；流水级数增多</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">过大</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>UB 占用增加，可能超出容量</strong>；流水的启动与排空阶段变长</td>
</tr>
</tbody>
</table>

因此存在一个折中点。下面在扫描的同时打印 UB 占用，对照 §3.1 的核算公式。

> 最后一档 8192 的占用为 3 × 8192 × 4 × 2 = 192 KB，**恰好等于 UB 的全部容量**。这是 §3.1 核算公式给出的可用上限，能否真正跑通还取决于编译器为临时变量留出多少空间，属于边界情况：可能编译或运行失败，也可能顺利通过。代码对失败做了捕获并单独标注，无论结果如何，这一档都应当计入实验结果一并分析。

In [ ]:
tile_lengths = [512, 1024, 2048, 4096, 8192]
scan_tile = []

for tl in tile_lengths:
    ub_kb = 3 * tl * 4 * 2 / 1024.0
    txt, ok = build_and_run({"LAB_TILE_LENGTH": tl}, tag="tl%d" % tl)
    if not ok:
        print(
            "❌ TILE_LENGTH=%-5d → UB 占用 %.1f KB，超出容量，编译或运行失败"
            "（这正是 §3.1 核算公式的意义）" % (tl, ub_kb)
        )
        scan_tile.append((tl, None, ub_kb, "FAIL"))
        continue
    r = [p for p in parse_perf(txt) if p["ver"] == "v3"]
    if r:
        scan_tile.append((tl, r[0]["kernel_ms"], ub_kb, parse_verify(txt).get("v3")))
        print(
            "TILE_LENGTH=%-5d UB=%6.1f KB (%4.1f%% of 192KB)  kernel_ms=%.4f  check=%s"
            % (
                tl,
                ub_kb,
                ub_kb / 192 * 100,
                r[0]["kernel_ms"],
                parse_verify(txt).get("v3"),
            )
        )

In [ ]:
ok_rows = [s for s in scan_tile if s[1] is not None]
if ok_rows:
    tl = [s[0] for s in scan_tile]
    ms = [s[1] for s in scan_tile]
    ref = min(s[1] for s in ok_rows)
    rel = [(ref / m) if m else 0.0 for m in ms]
    cols = [C_KERNEL if m else "#4d4d4d" for m in ms]

    fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=120)
    ax.bar(
        [str(t) for t in tl],
        rel,
        0.55,
        color=cols,
        hatch=["" if m else "//" for m in ms],
    )
    for i, (m, r_) in enumerate(zip(ms, rel)):
        ax.text(
            i,
            r_ if m else 0.02,
            ("%.2fx" % r_) if m else "UB overflow",
            ha="center",
            va="bottom",
            fontsize=8,
            color="black" if m else "#4d4d4d",
            rotation=0 if m else 90,
        )
    ax.set_xlabel("TILE_LENGTH (elements)")
    ax.set_ylabel("Relative throughput (best = 1.0)")
    ax.set_title("Lab 2: tile-length sweep (v3)")
    ax.grid(axis="y", alpha=0.3)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.show()

### 12.3 规模扫描

最后一组扫描不需要重新编译，直接使用 v4 的运行时长度参数即可。这一组数据用于考察**加速比随问题规模的变化趋势**：小规模下核函数的启动与调度开销占主导，规模增大后固定开销被摊薄，加速比随之上升，并在带宽饱和处趋于平缓。解读时须同时对照 `cpu_ms` 与 `kernel_ms` 两列原始耗时，原因见 §13 ④。

In [ ]:
sizes = [8 * 1024, 64 * 1024, 512 * 1024, 2 * 1024 * 1024, 8 * 1024 * 1024]
scan_n = []
for nn in sizes:
    txt = run_demo([nn])
    r = [p for p in parse_perf(txt) if p["ver"] == "v4"]
    if r:
        scan_n.append(r[0])
        print(
            "N=%-9d cpu_ms=%9.4f kernel_ms=%8.4f  vsCPU(kernel)=%7.2fx"
            % (nn, r[0]["cpu_ms"], r[0]["kernel_ms"], r[0]["sp_kernel"])
        )

In [ ]:
if scan_n:
    N = [r["n"] for r in scan_n]
    sk = [r["sp_kernel"] for r in scan_n]
    km = [r["kernel_ms"] for r in scan_n]

    fig, ax = plt.subplots(figsize=(7.4, 4.3), dpi=120)
    ax.plot(N, sk, marker="o", lw=2, color=C_KERNEL, label="speedup vs. CPU (kernel)")
    ax.axhline(1.0, color="#888888", lw=1.0, ls=":")
    ax.text(N[0], 1.06, "baseline: CPU = 1.0x", fontsize=9, color="#777777")
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("Problem size N (elements)")
    ax.set_ylabel("Speedup over CPU baseline")

    # 同时给出原始核函数耗时：判断 NPU 侧是否已接近带宽上限，须看它随 N 是否趋于线性增长，
    # 仅凭加速比无法区分 NPU 侧变快与 CPU 基准变慢这两种来源（见 §13 ④）。
    ax2 = ax.twinx()
    ax2.plot(N, km, marker="s", lw=2, ls="--", color=C_ALT, label="kernel_ms (raw)")
    ax2.set_yscale("log")
    ax2.set_ylabel("kernel_ms", color=C_ALT)

    ax.set_title("Lab 2: kernel speedup and raw kernel time vs. problem size (v4)")
    ax.grid(alpha=0.3, which="both")
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, frameon=False, loc="upper left")
    ax.spines["top"].set_visible(False)
    plt.tight_layout()
    plt.show()

## 13. 结果分析

> 以下结论针对**趋势规律**。具体数值随硬件规格、CANN 版本与系统负载而变化。

**① v1 → v2：多核切分带来的加速最为显著**

数据被切分到 8 个核上，各核的工作量降为原来的八分之一。这是本实验中收益最大的一步，其性质与第五章 OpenMP 实验中从串行到并行的那一步相同。

实测加速比通常**低于**核数。原因在于核函数的启动与调度开销不随核数减少，且各核共享同一条 Global Memory 通路——切分数据并不会使总的访存量下降。

**② v2 → v3：双缓冲的收益取决于搬运与计算的时间比例**

双缓冲的作用是让两类部件并发工作，其收益上限由二者中较慢的一方决定。向量加法是**访存密集型**算子：每处理一个元素需要读 2 个 float、写 1 个 float，共 12 字节，而计算只有 1 次浮点加法。搬运时间远大于计算时间，因此双缓冲的收益虽然存在，但通常不如 v1 → v2 那一步显著。

> **一处需要说明的因素**：本实验的 v3 相对 v2 把 UB 占用从 48 KB 提高到了 96 KB（见 §2.4）。这一步的收益中，既有「搬运与计算重叠」的贡献，也有「可用缓冲变多」的贡献，两者并未分离。若要做等 UB 预算的对照，应当把 v3 的 `TILE_LENGTH` 减半（2048 × 2 块 = 48 KB）再与 v2 比较。§12.2 的分块长度扫描恰好提供了该对照所需的另一半数据：v3 在 `TILE_LENGTH = 2048` 时的耗时可直接与 v2 在 4096 时的耗时相比。需要注意二者的单次搬运粒度并不相同，因此这一对照仍不完全干净。动手练习第 1 题要求补齐并解释该结果。

**③ v3 → v4：尾块判断的开销可以忽略**

v4 相对 v3 只增加了一次分支判断与至多一次额外的分块处理，且都在核内只执行一次。若二者耗时接近，说明泛化能力的取得几乎没有付出性能代价——并非所有的正确性改进都以性能为代价。

**④ CPU 对比：kernel-only 加速比的读法，以及卸载收益应当如何判断**

kernel-only 加速比在最小的几档规模下可能小于 1，此时核函数的启动与调度开销占主导；规模增大后固定开销被摊薄，加速比随之上升，并在访存带宽饱和处趋于平缓。

这里有一处需要提醒：**加速比是一个比值，它的上升既可能来自 NPU 侧变快，也可能来自 CPU 基准变慢**。当数据量超出 CPU 末级缓存后，CPU 侧每元素的耗时会明显上升，仅凭加速比曲线无法区分这两种来源。要判断 NPU 侧是否已接近带宽上限，应当直接考察 `kernel_ms` 随 N 的增长是否趋于线性，而不能只看加速比。§12.3 的图同时给出加速比与原始 `kernel_ms`，正是为此。

至于单个访存受限算子是否值得放到 NPU 上执行这一问题，本实验不通过端到端计时回答，理由见 §5.1。从算术强度出发即可得到判断：本算子每处理一个元素需要在主机与设备之间往返搬运 12 字节，而计算只有 1 次浮点加法，搬运量与计算量同阶且都随 N 线性增长。若为它单独安排一次完整的主机—设备往返，可被加速的部分在总时间中占比极低，按第一章的 **Amdahl 定律**，整体收益必然有限；而搬运本身的耗时又取决于 CPU 访存带宽与主机—设备链路带宽这两个平台参数，同一段代码在不同机器上会得到不同的结论。因此这类端到端比值既无法给出普遍结论，也没有对应的真实用法。

> **工程结论**：不宜为单个访存受限的算子在主机与设备之间往返搬运数据。真实的推理框架让数据一次搬入后常驻设备内存，由连续的多个算子接力处理，中间结果不回传主机，并进一步把相邻算子**融合**为一个核函数，用一次搬运摊薄多个算子的计算量。**是否卸载到 NPU，判断单位是整段计算图，而不是单个算子。**这就是实验五要讨论的算子融合。

**⑤ 加速比必须与基准耗时一并报告**

上一条提示的问题值得单独强调。在两台配置不同的机器上运行本实验，`kernel_ms` 可能十分接近，而相对 CPU 的加速比却相差数倍——差异全部来自 CPU 基准的快慢，与 NPU 无关。

由此得到一条通用的实验规范：**加速比由分子与分母共同决定，脱离基准的加速比没有意义**。本实验的 `[PERF]` 记录行同时输出 `cpu_ms` 与 `kernel_ms` 两个原始耗时，正是为了让读者能够还原比值的来源。报告性能结论时，应当同时给出基准实现、硬件型号与测量口径。

**⑥ 核数扫描：加速比偏离线性的两种成因**

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 成因 | 判别方式 |
| --- | --- |
| 启动与调度的固定开销 | 减小 N 后，偏离线性在更低的核数处即已出现 |
| Global Memory 带宽饱和 | 增大 N 后，偏离线性仍出现在相近的核数处 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">成因</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">判别方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">启动与调度的固定开销</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">减小 N 后，偏离线性在更低的核数处即已出现</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Global Memory 带宽饱和</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">增大 N 后，偏离线性仍出现在相近的核数处</td>
</tr>
</tbody>
</table>

加速比在各档均低于核数，自第一次翻倍起即出现偏离，且偏离程度随核数增大而扩大。做出判别需要在不同的 N 下各做一次核数扫描，动手练习第 4 题即为该对照；§12.3 的规模扫描固定在同一核数上运行，不足以完成这一判别。

**⑦ 分块长度扫描：折中点与 UB 容量这条边界**

分块过小时，单次搬运的有效载荷不足，搬运指令的固定开销占比上升，耗时随分块长度减小而显著增加。分块增大到一定程度后收益趋于平缓：此时搬运已能较充分地利用带宽，继续增大主要是把流水的启动与排空阶段拉长。**最优分块长度落在哪一档与平台相关，应由实测确定**，既不应假定越大越好，也不应假定最大档一定劣于中间档。

`TILE_LENGTH = 8192` 一档的 UB 占用为 192 KB，恰好等于 §3.1 核算公式给出的容量上限。这一档能否跑通取决于编译器还需要为临时变量留出多少空间，属于边界情况。无论结果如何，这一档要读出的结论是：**UB 容量是一条硬约束**，§3.1 的核算公式给出的是可用上限而非安全取值；片上缓冲的分配应当留有余量，最优取值由实测确定。

**⑧ 从 Roofline 模型看本算子的位置**

本算子的算术强度为 I = 1 FLOP / 12 Byte ≈ 0.083 FLOP/Byte。这一数值极低，位于 Roofline 模型中远离屋脊的斜坡段，即典型的访存受限区域。由此可以推断两条结论：

- 任何**减少访存量**的手段（如**算子融合**）都将直接带来收益；
- 任何**提升计算能力**的手段（如换用更强的计算单元）都不会带来收益。

这正是下一步讨论算子融合的动因，也解释了 ④ 中 kernel 加速比最终会饱和的原因：限制它的是访存带宽，而不是计算能力。

同一套分析也适用于 CPU 基准：单线程逐元素加法同样受访存带宽限制。因此 ④ 中的 kernel-only 加速比，本质上是两侧可调动的访存带宽之比，而不是计算能力之比——这也从机制上解释了 ⑤ 中加速比随基准而变的现象。由此还可以理解单核版本 v1 相对 CPU 的加速比为何并不高：一个 AI Core 能调动的带宽与一个 CPU 核相差有限，真正拉开差距的是多核并行之后可以并发发起的访存请求数量。

---

### 🎓 结论

向量加法揭示了昇腾算子开发的三层结构：**核间切分决定并行度，核内分块使数据得以放入片上缓冲，流水与双缓冲决定核内部件的利用率。** 三者缺一不可，且各自受不同的约束支配。

本实验的算术强度分析与 CPU 对比给出了两条更普遍的判断：**当算子处于访存受限区域时，优化的方向应当是减少访存量，而非提高计算效率；卸载到 NPU 是否划算，判断单位是整段计算图而非单个算子——数据一次搬入后应尽可能长时间地留在设备内存中，由连续的多个算子接力处理。**

## 14. 🔧 动手练习

> **提示**：修改源码需要从 §7.1 开始按顺序重新执行全部写入单元格。多数练习只需修改编译参数，用 `build_and_run({'LAB_XXX': 值})` 一行即可完成。

1. **等 UB 预算的双缓冲对照**（用于修正 §13 ② 指出的比较偏差）。用 `build_and_run({'LAB_TILE_LENGTH': 2048})` 重新编译，此时 v3 的 UB 占用为 48 KB，与 v2 在 `TILE_LENGTH=4096` 时相同。比较这两个配置下 v3 与 v2 的耗时，说明双缓冲**纯粹的**重叠收益是多少。

2. **v3 处理非整除长度**。把 §7.3 的 `static_assert` 临时注释掉，把 v4 的调用改为 v3，用 `N_ODD` 运行，记录**超差元素的数量与首个超差位置**。然后回答：为什么超差不是从第 2 097 152 个元素才开始？*提示：检查各核的 GM 起始地址 262269 × 4 = 1 049 076 字节是否为 32 的整数倍。* 该题说明对齐约束的违反比「漏算尾部」严重得多。

3. **交错切分与连续切分的对照**。把 v2 的切分方式由「连续分片」改为「按分块交错」，即第 k 个核处理序号模核数等于 k 的全部分块。校验是否仍然通过？性能如何变化？请结合搬运地址的连续性解释。

4. **判别加速比平缓的成因**。用 `build_and_run({'LAB_BLOCK_DIM': bd, 'LAB_TOTAL_LENGTH': N})` 在 N = 2^18 与 N = 2^23 两个规模下各做一次核数扫描，对比两条曲线的平缓点位置，据此判断 §13 ⑥ 中哪一种成因占主导。

5. **【进阶】改用 half**。把 `using LabDType = float;` 改为 `half`。需要连带修改的地方有四处：`ALIGN_ELEM` 由 8 变为 **16**、UB 占用减半、算术强度翻倍、校验容差需放宽到 1e-3 量级。逐一核算后再实测，说明性能提升了多少、原因是什么。

6. **【进阶】测量核函数的启动开销**。当前的 `TIME_KERNEL` 每轮都同步一次，测得的是单次调用的完整时延。再写一个变体：`REPEAT` 次启动连续下发、循环结束后只同步一次，测得的是流水吞吐下的摊薄时间。两者之差即被流水掩盖掉的启动开销，可用它量化 §13 ⑥ 的第一种成因。

## 15. 🤔 思考题

1. `Compute` 中的 `FreeTensor` 若移到 `CopyOut` 之后再调用，正确性是否受影响？性能是否受影响？请结合队列的同步语义作答。

2. `AllocTensor` 在队列中没有空闲缓冲时会发生什么？这与第四章有界缓冲区中生产者遇到缓冲区满时的行为有何异同？

3. 双缓冲使 UB 占用翻倍。若改为三缓冲，收益与代价将如何变化？在什么条件下继续增加缓冲块数不再有效？

4. v2 中若把偏移改在每次 `DataCopy` 时叠加，而不写入 `SetGlobalBuffer`，程序是否仍然正确？这样写会带来什么问题？

5. v4 要求元素总数为 8 的整数倍。带填充能力的搬运接口 **`DataCopyPad`**（配合 `DataCopyExtParams` 与 `DataCopyPadExtParams` 两个参数结构）可以放宽这一约束。请查阅文档说明它如何处理非 32 字节对齐的尾块，并给出改造 v4 的方案。

6. 第五章讨论过的**伪共享**，在本实验中会不会出现？为什么？*（提示：各核的 UB 是否共享同一条缓存行？各核写入 GM 的区域是否相邻？）*

7. 若一个应用需要连续做 100 次向量加法，每次的输入都是上一次的输出，应当如何组织主机与设备之间的数据搬运才能使 NPU 产生实际收益？请分别估算「每次都完整往返一遍」与「只在首尾各搬运一次」两种组织方式下，主机与设备之间的搬运总量之比。*（该问题将在实验五中系统回答。）*

8. 本实验的 `TIME_KERNEL` 每轮都同步。若改为下发 50 次后只同步一次，测得的数值会变大还是变小？它更接近时延还是吞吐？在什么场景下应当报告前者，什么场景下应当报告后者？

## 16. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| 片上缓冲 | 矢量单元只能访问 UB（192 KB），数据必须显式搬入搬出；未搬入则**无法计算** |
| 四类对象 | `GlobalTensor` 描述 GM 数据，`LocalTensor` 描述片上数据，`TPipe` 管内存，`TQue` 管流水 |
| 三段流水 | CopyIn / Compute / CopyOut，分别对应 MTE2、VEC、MTE3 三条独立指令队列 |
| 队列的双重职责 | `EnQue` 通知下游数据就绪，`FreeTensor` 通知上游缓冲可复用；两者都必须成对 |
| 任务类型声明 | 同一编译单元存在多个核函数时不自动推导 Kernel 类型，须用 `KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY)` 显式声明 |
| SPMD 切分 | 依据 `block_idx` 计算偏移，写入 `SetGlobalBuffer` 使核内其余代码保持不变 |
| 两级切分 | 核间切分决定并行度，核内分块受 UB 容量约束 |
| UB 占用核算 | 3 × TILE_LENGTH × sizeof(T) × BUFFER_NUM，编写代码之前应先完成核算 |
| 双缓冲 | 以 UB 占用翻倍为代价，换取搬运与计算的重叠 |
| 对齐约束 | 搬运的起始地址与长度均须为 32 字节的整数倍（float 即 8 个元素） |
| 尾核与尾块 | 官方的尾核指数据无法均分时计算量较少的那组核；本实验采用末核承担余量的简化方案。尾块指核内不足一个完整分块的部分 |
| 性能测量 | 预热、多次取平均、**先同步再停止计时**；报告加速比时必须一并给出基准耗时 |
| 基准实现 | 与被测实现使用相同优化级别，且其结果必须被后续代码使用 |
| 校验方式 | 随机数据下使用**相对误差**，不能使用位级比对 |
| 算术强度 | 本算子为 1 FLOP / 12 Byte，属访存受限，优化方向是**减少访存量** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">片上缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矢量单元只能访问 UB（192 KB），数据必须显式搬入搬出；未搬入则<strong>无法计算</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">四类对象</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GlobalTensor</code> 描述 GM 数据，<code>LocalTensor</code> 描述片上数据，<code>TPipe</code> 管内存，<code>TQue</code> 管流水</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三段流水</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CopyIn / Compute / CopyOut，分别对应 MTE2、VEC、MTE3 三条独立指令队列</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">队列的双重职责</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>EnQue</code> 通知下游数据就绪，<code>FreeTensor</code> 通知上游缓冲可复用；两者都必须成对</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">任务类型声明</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同一编译单元存在多个核函数时不自动推导 Kernel 类型，须用 <code>KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY)</code> 显式声明</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">SPMD 切分</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">依据 <code>block_idx</code> 计算偏移，写入 <code>SetGlobalBuffer</code> 使核内其余代码保持不变</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两级切分</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核间切分决定并行度，核内分块受 UB 容量约束</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">UB 占用核算</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3 × TILE_LENGTH × sizeof(T) × BUFFER_NUM，编写代码之前应先完成核算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">双缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">以 UB 占用翻倍为代价，换取搬运与计算的重叠</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">对齐约束</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">搬运的起始地址与长度均须为 32 字节的整数倍（float 即 8 个元素）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">尾核与尾块</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">官方的尾核指数据无法均分时计算量较少的那组核；本实验采用末核承担余量的简化方案。尾块指核内不足一个完整分块的部分</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">性能测量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">预热、多次取平均、<strong>先同步再停止计时</strong>；报告加速比时必须一并给出基准耗时</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基准实现</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与被测实现使用相同优化级别，且其结果必须被后续代码使用</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">校验方式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">随机数据下使用<strong>相对误差</strong>，不能使用位级比对</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算术强度</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">本算子为 1 FLOP / 12 Byte，属访存受限，优化方向是<strong>减少访存量</strong></td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **在异构算子开发中，性能问题首先是数据摆放问题，其次才是计算问题。**

v1 到 v4 的四个版本中，没有任何一处改动涉及计算本身——`AscendC::Add` 的调用形式自始至终没有变化。全部的性能差异都来自数据如何切分、如何搬运、以及搬运与计算如何重叠。

### 与后续实验的衔接

➡️ **后续内容：实验三 · 规约算子 ReduceSum**。本实验的每个输出元素只依赖对应位置的输入，各核之间完全独立。下一个实验将引入**规约**：输出元素依赖全部输入，各核的部分结果必须合并。届时将出现本实验刻意回避的问题——**核间的数据交换**，以及为此引入的 workspace 与原子累加机制。